In [23]:
import pandas as pd

df = pd.read_csv("/content/datos_tratados.csv")

print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (7032, 22)


,customerID,Churn,customer_gender,customer_SeniorCitizen,customer_Partner,customer_Dependents,customer_tenure,phone_PhoneService,phone_MultipleLines,internet_InternetService,...,internet_DeviceProtection,internet_TechSupport,internet_StreamingTV,internet_StreamingMovies,account_Contract,account_PaperlessBilling,account_PaymentMethod,account_Charges_Monthly,account_Charges_Total,Cuentas_Diarias
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.30,2.186667
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.40,1.996667
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85,2.463333
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85,3.266667
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.40,2.796667


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customerID                 7032 non-null   object 
 1   Churn                      7032 non-null   object 
 2   customer_gender            7032 non-null   object 
 3   customer_SeniorCitizen     7032 non-null   int64  
 4   customer_Partner           7032 non-null   object 
 5   customer_Dependents        7032 non-null   object 
 6   customer_tenure            7032 non-null   int64  
 7   phone_PhoneService         7032 non-null   object 
 8   phone_MultipleLines        7032 non-null   object 
 9   internet_InternetService   7032 non-null   object 
 10  internet_OnlineSecurity    7032 non-null   object 
 11  internet_OnlineBackup      7032 non-null   object 
 12  internet_DeviceProtection  7032 non-null   object 
 13  internet_TechSupport       7032 non-null   objec

In [24]:
df['Churn'].value_counts(dropna=False)

,count
Churn,
No,5163
Yes,1869


In [25]:
#Preparación: target binario + drop de ID
df = df.copy()

# target
df["Churn_bin"] = df["Churn"].map({"No": 0, "Yes": 1})

# eliminar columnas irrelevantes
df = df.drop(columns=["customerID", "Churn"])

df["Churn_bin"].value_counts(normalize=True).round(4)

,proportion
Churn_bin,
0,0.7342
1,0.2658


In [26]:
#Separar X / y + One-Hot Encoding
X = df.drop(columns=["Churn_bin"])
y = df["Churn_bin"]

X_enc = pd.get_dummies(X, drop_first=True)

X_enc.shape, y.shape

((7032, 31), (7032,))

In [27]:
#Train/Test Split (estratificado)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape


((5625, 31), (1407, 31))

In [28]:
#Modelo 1 (con normalización): Regresión Logística
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

log_reg = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

log_reg.fit(X_train, y_train)


Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=2000))])

In [29]:
#Modelo 2 (sin normalización): Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [30]:
#Evaluación (accuracy, precision, recall, f1 + matriz confusión)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

def eval_model(model, X_test, y_test, name="Modelo"):
    y_pred = model.predict(X_test)
    print(f"=== {name} ===")
    print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred), 4))
    print("Recall   :", round(recall_score(y_test, y_pred), 4))
    print("F1-score :", round(f1_score(y_test, y_pred), 4))
    print("\nMatriz de confusión:")
    print(confusion_matrix(y_test, y_pred))
    print("\nReporte:")
    print(classification_report(y_test, y_pred))

eval_model(log_reg, X_test, y_test, "Regresión Logística")
eval_model(rf, X_test, y_test, "Random Forest")

=== Regresión Logística ===
Accuracy : 0.801
Precision: 0.6556
Recall   : 0.5294
F1-score : 0.5858

Matriz de confusión:
[[929 104]
 [176 198]]

Reporte:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1033
           1       0.66      0.53      0.59       374

    accuracy                           0.80      1407
   macro avg       0.75      0.71      0.73      1407
weighted avg       0.79      0.80      0.79      1407

=== Random Forest ===
Accuracy : 0.7903
Precision: 0.6479
Recall   : 0.4626
F1-score : 0.5398

Matriz de confusión:
[[939  94]
 [201 173]]

Reporte:
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1033
           1       0.65      0.46      0.54       374

    accuracy                           0.79      1407
   macro avg       0.74      0.69      0.70      1407
weighted avg       0.78      0.79      0.78      1407



In [31]:
#Importancia de variables (RF) + Coeficientes (LogReg)
import pandas as pd

# Random Forest - top 15
rf_imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
rf_imp

,0
customer_tenure,0.153698
account_Charges_Total,0.152903
Cuentas_Diarias,0.119104
account_Charges_Monthly,0.116278
account_Contract_Two year,0.054309
account_PaymentMethod_Electronic check,0.038359
internet_InternetService_Fiber optic,0.038351
account_Contract_One year,0.028376
customer_gender_Male,0.023352
internet_OnlineSecurity_Yes,0.023325


In [33]:
# Logistic Regression - top 15 (por valor absoluto)
coef = log_reg.named_steps["model"].coef_[0]
log_imp = pd.Series(coef, index=X_train.columns).abs().sort_values(ascending=False).head(15)
log_imp

,0
customer_tenure,1.313607
internet_InternetService_Fiber optic,0.644955
account_Charges_Total,0.609028
account_Contract_Two year,0.565938
Cuentas_Diarias,0.336935
account_Charges_Monthly,0.336935
account_Contract_One year,0.290447
internet_StreamingTV_Yes,0.240530
account_PaperlessBilling_Yes,0.181353
internet_StreamingMovies_Yes,0.170661


# **Informe final - Telecom X (Parte 2): Predicción de Cancelación (Churn)**

# Introducción
El objetivo de este proyecto es desarrollar modelos predictivos capaces de estimar la probabilidad de cancelación de clientes (Churn) en Telecom X. La empresa presenta una tasa relevante de evasión y requiere anticiparse al riesgo de pérdida de clientes para priorizar acciones de retención, reducir el impacto financiero y orientar estrategias comerciales basadas en evidencia.

# Preparación y tratamiento de datos
Se utilizó como base el dataset tratado en la Parte 1 del desafío (CSV exportado desde el proceso ETL). En esta segunda etapa se realizaron pasos orientados al modelamiento:
* Carga del dataset tratado y validación inicial de estructura (dimensiones, tipos y consistencia de columnas).
* Definición de variable objetivo: se creó Churn_bin (0 = No cancela, 1 = Cancela), manteniendo Churn como referencia inicial.
* Eliminación de columnas irrelevantes para el modelado: se removieron identificadores únicos como customerID y se excluyó la columna objetivo original Churn para evitar fugas de información.
* Codificación de variables categóricas mediante One-Hot Encoding (pd.get_dummies(..., drop_first=True)), transformando el conjunto de variables a formato numérico apto para algoritmos de Machine Learning.
* Separación de datos en entrenamiento y prueba usando train_test_split con estratificación, preservando la proporción de churn entre ambos conjuntos.
* Normalización/estandarización: aplicada solo al modelo sensible a escala (Regresión Logística) mediante un Pipeline con StandardScaler. Para Random Forest no fue necesaria, por tratarse de un modelo basado en árboles.

# Distribucion y proporción de cancelación
La proporción observada en el dataset indica un escenario de desbalance moderado, con mayor presencia de clientes no canceladores. Esto es relevante porque métricas como accuracy pueden resultar optimistas si no se analizan conjuntamente con recall, precisión y F1-score de la clase minoritaria (churn).

# Modelado predictivo
Se entrenaron y compararon dos modelos:
1. Regresión Logística (con estandarización): modelo lineal interpretable y adecuado como baseline fuerte para clasificación binaria.
2. Random Forest (sin estandarización): modelo no lineal, robusto, capaz de capturar interacciones y relaciones complejas entre variables.

Ambos fueron evaluados en el conjunto de prueba con:
* Accuracy
* Precisión
* Recall
* F1-score
* Matriz de confusión
* Reporte de clasificación

# Evaluación y comparación de desempeño

Los resultados muestran que ambos modelos obtienen desempeños comparables a nivel general, pero con diferencias en su comportamiento frente a la clase positiva (clientes que cancelan):
* La Regresión Logística presenta un desempeño competitivo y tiende a mantener interpretabilidad más directa.
* El Random Forest ofrece buen desempeño global y una lectura clara de importancia de variables, aunque puede requerir calibración/ajuste de hiperparámetros si se busca mejorar el recall del churn (capturar más canceladores).

Dado el objetivo de negocio (reducir evasión), suele ser preferible priorizar recall de la clase churn (detectar más clientes con riesgo), incluso si ello implica un aumento controlado de falsos positivos.

# Interpretación: variables más relevantes

Se interpretaron variables relevantes de dos maneras:
* Random Forest: importancia por reducción de impureza.
* Regresión Logística: magnitud de coeficientes (en valor absoluto), interpretados como influencia relativa en la predicción.

En ambos enfoques, se repiten como factores clave:
* Antigüedad del cliente (customer_tenure): variable con mayor asociación. Clientes con menor permanencia tienden a presentar más riesgo de churn.
* Variables de gasto (account_Charges_Total, account_Charges_Monthly, Cuentas_Diarias): reflejan comportamiento de consumo y nivel de facturación. Diferencias de gasto se asocian con cambios en riesgo.
* Tipo de contrato (account_Contract_Two year, account_Contract_One year): contratos más largos se relacionan con menor churn (mayor retención).
* Internet Fibra Óptica (internet_InternetService_Fiber optic): aparece como variable relevante, sugiriendo que el tipo de servicio o la experiencia asociada puede influir en cancelación.
* Método de pago (ej. account_PaymentMethod_Electronic check): puede reflejar patrones de comportamiento y perfiles con distinta propensión a evasión.
* Variables de servicios complementarios (seguridad online, soporte técnico, streaming) también contribuyen, aunque con menor peso relativo.

# Conclusiones e insights estratégicos

1. La permanencia es el predictor más consistente: clientes con menor tenure concentran el riesgo, por lo que deben ser el foco de estrategias preventivas tempranas (onboarding, seguimiento de primeros meses, ofertas personalizadas).
2. Contratos largos funcionan como mecanismo de retención: planes de 1–2 años reducen churn. Incentivar migración desde “month-to-month” hacia contratos largos puede mejorar retención.
3. Patrones de gasto y facturación se asocian al churn: cambios en costo mensual o percepción de valor pueden disparar cancelaciones; es recomendable monitorear segmentos con cobros altos o variabilidad.
4. Servicios y tipo de internet importan: la fibra óptica y servicios asociados podrían estar vinculados a experiencia del cliente (calidad, precio o soporte). Conviene investigar causas específicas (reclamos, performance, pricing).

# Recomendaciones

* Crear campañas focalizadas en clientes nuevos (bajo tenure) con alertas tempranas y beneficios por permanencia.
* Diseñar estrategias para aumentar la adopción de contratos anuales/bianuales (descuentos, bundles, beneficios escalonados).
* Implementar monitoreo de riesgo churn con umbrales basados en probabilidad, priorizando recall y ajustando el umbral según capacidad operativa del área de retención.
* Profundizar análisis de segmentos de fibra óptica: revisar calidad del servicio, incidencias y percepción de valor.
* Iterar el modelo con optimización de hiperparámetros y, si el desbalance aumenta, evaluar técnicas como class_weight, SMOTE o calibración.
